# Заполнить таблицу

# Python. Семинар 6

Подготовка к семинару «Классы вблизи»: короткий recap лекции 6, углубление
с фишками и викторина «предскажи, что выведет». Разборы вопросов спрятаны
под спойлер — сначала ответь сам, потом раскрой.

---

## Часть 1. Быстрый recap лекции

### 1.1. Статичный метод и фабрики

`@staticmethod` крепит метод к классу: он лежит в `Класс.__dict__`, но не
принимает ни `self`, ни `cls`. Главное применение — **фабрики**,
альтернативные конструкторы: разобрать строку и вернуть объект, собранный
обычным `__init__`:

In [ ]:
class Date:
    def __init__(self, day: int, month: int, year: int) -> None:
        self.day, self.month, self.year = day, month, year

    def __str__(self) -> str:
        return f"{self.day:02d}.{self.month:02d}.{self.year}"

    @staticmethod
    def from_iso_format(iso_string: str) -> "Date":
        year, month, day = map(int, iso_string.split("-"))
        return Date(day, month, year)

    @staticmethod
    def from_american_format(us_string: str) -> "Date":
        month, day, year = map(int, us_string.split("/"))
        return Date(day, month, year)


print(Date(25, 12, 2023))
print(Date.from_iso_format("2023-12-25"))
print(Date.from_american_format("12/25/2023"))

Один класс — три способа создания. Фабрика не имеет доступа к объекту (его
ещё нет) и потому возвращает результат обычного вызова `Date(...)`. Подробнее —
2.10.

### 1.2. Наследование и `super()`

Наследник получает все методы родителей: своего нет — поиск уходит вверх до
`object`. Переопределяя метод, родительскую версию вызывают через `super()`:

In [ ]:
class User:
    def __init__(self, name: str) -> None:
        self.name = name
        print(f"{self.name} родился!")


class Admin(User):
    def __init__(self, name: str) -> None:
        super().__init__(name)          # сначала — родительский __init__
        print(f"{self.name} вылупился!")


Admin("Птенец")

Родитель наполовину сделал работу, наследник дополнил. То же самое одной
строкой — `User.__init__(self, name)` — работает, но `super()` не привязан к
жёсткому имени класса.

### 1.3. MRO: порядок поиска метода

Какого родителя спросить первым, зафиксировано в `__mro__` — Method Resolution
Order. Список можно посмотреть у любого класса:

In [32]:
class Reader:
    def greet(self) -> str:
        return "привет от Reader"


class Writer:
    def greet(self) -> str:
        return "привет от Writer"


class ReadWriter(Reader, Writer):
    pass


print([cls.__name__ for cls in ReadWriter.__mro__])
print(ReadWriter().greet())

['ReadWriter', 'Reader', 'Writer', 'object']
привет от Reader


Поиск шёл слева направо: `Reader` ответил первым, до `Writer` дело не дошло.
Каждый класс встречается в `__mro__` один раз — ромбовидное наследование не
зацикливает поиск.

### 1.4. `__slots__`

`__slots__` — кортеж имён полей, который заменяет объекту персональный
`__dict__`. Объект худеет (на сотнях тысяч инстансов это проценты памяти),
но завести поле вне списка уже нельзя:

In [33]:
class Point:
    __slots__ = ("x", "y")

    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y


point = Point(1, 2)
print(hasattr(point, "__dict__"), point.__slots__)

try:
    point.z = 3
except AttributeError as e:
    print("AttributeError:", e)

False ('x', 'y')
AttributeError: 'Point' object has no attribute 'z' and no __dict__ for setting new attributes


Плата за экономию — жёсткость. И главное: слоты действуют только на свой
класс, у наследника без объявления всё возвращается на круги своя — 2.6.

### 1.5. Перегрузка операторов

`+`, `*`, `+=` и печать — это методы: `__add__`, `__mul__`, `__iadd__`,
`__str__`. Если метод не понял операнд, он возвращает `NotImplemented` — не
исключение, а сигнал «спроси второго»:

In [34]:
from typing import Self


class Number:
    def __init__(self, x: int) -> Self:
        self.x = x

    def __str__(self) -> str:
        return f"Number({self.x})"

    def __add__(self, other: object) -> Self:
        if isinstance(other, Number):
            return Number(self.x + other.x)
        return NotImplemented            # «не умею» — но не падение


print(Number(2) + Number(3))

try:
    Number(2) + "мимо"
except TypeError as e:
    print("TypeError:", e)

Number(5)
TypeError: unsupported operand type(s) for +: 'Number' and 'str'


Оба отказались — Python поднял `TypeError`. Теперь вариант с приставкой `i`:
`__iadd__` для `+=` меняет объект на месте и возвращает `self`, обычный `+`
строит новый объект. Свидетель — второе имя на тот же объект:

In [35]:
from typing import Self


class Number:
    def __init__(self, x: int) -> Self:
        self.x = x

    def __add__(self, other: "Number") -> "Number":
        return Number(self.x + other.x)   # новый объект

    def __iadd__(self, other: "Number") -> "Number":
        self.x += other.x                 # мутация на месте
        return self                       # правило всех i-методов

    def __str__(self) -> str:
        return f"Number({self.x})"


a = Number(6)
alias = a
a += Number(4)
print(a, alias, a is alias)

a = a + Number(4)
print(a, alias, a is alias)

Number(10) Number(10) True
Number(14) Number(10) False


Разница — одна буква в методе, последствия — противоположные (подробнее 2.4).
Умножение добавляет reflected-механику: `a * 3` идёт в `__mul__`, а `3 * a` —
число про нас не знает, и Python вызывает наш `__rmul__`. Рядом `__call__` —
метод, который превращает объект в функцию:

In [36]:
from typing import Self


class Number:
    def __init__(self, x: int) -> Self:
        self.x = x

    def __mul__(self, other: int) -> Self:
        print("__mul__")
        return Number(self.x * other)

    def __rmul__(self, other: int) -> Self:
        print("__rmul__")
        return Number(other * self.x)

    def __str__(self) -> str:
        return f"Number({self.x})"


print(Number(6) * 3)
print(3 * Number(6))


class Power:
    def __init__(self, p: float) -> None:
        self.p = p

    def __call__(self, a: float) -> float:
        return a**self.p


cube = Power(3)
print(cube(4))

__mul__
Number(18)
__rmul__
Number(18)
64


`cube(4)` — это `cube.__call__(4)`: объект вызывается как функция. Кто
вызывался — `__mul__` или `__rmul__` — разбираем в 2.5.

### 1.6. `property`

`property` управляет доступом к атрибуту: чтение и запись уходят в методы, а
снаружи всё выглядит как обычное поле. Классика — проверить значение перед
записью (пример с лекции):

In [ ]:
from typing import Self


class LineItem:
    def __init__(self, description: str, weight: float, price: int) -> Self:
        self.description = description
        self.weight = weight            # не поле, а вызов set_weight
        self.price = price

    def subtotal(self) -> float:
        return self.weight * self.price

    def get_weight(self) -> float:
        return self.__weight

    def set_weight(self, value: float) -> None:
        if value > 0:
            self.__weight = value
        else:
            raise ValueError("value must be > 0")

    weight = property(fget=get_weight, fset=set_weight)


item = LineItem("Чемодан", 3.4, 300)
print(item.weight, item.subtotal())

try:
    item.weight = -1
except ValueError as e:
    print("ValueError:", e)

Обратите внимание на `self.weight = weight` в `__init__`: конструктор тоже
идёт через сеттер, поэтому и ему проверка не обойти. Поле под капотом —
`self.__weight` с двумя подчёркиваниями; что это значит — прямо сейчас.

### 1.7. Контроль доступа

Приватности в Python нет. Двойное подчёркивание не запрещает доступ, а лишь
переименовывает атрибут в `_ИмяКласса__атрибут` — это name mangling.
Договорённость вместо запретов — одно подчёркивание `_`: «не трогать снаружи»:

In [37]:
from typing import Self


class Vault:
    def __init__(self, secret: str) -> Self:
        self.__secret = secret          # на деле: _Vault__secret


vault = Vault("пароль")

try:
    print(vault.__secret)
except AttributeError as e:
    print("AttributeError:", e)

print(vault._Vault__secret)             # а достать можно всегда

AttributeError: 'Vault' object has no attribute '__secret'
пароль


Правила лекции: «`__` не обеспечивает приватность», «никогда не используйте
двойное подчёркивание в нейминге атрибутов». Поэтому в 1.6 поле под капотом —
это деталь реализации, а не защита.

### 1.8. `@dataclass` и `Enum`

`@dataclass` генерирует служебные методы: `__init__`, `__repr__`, `__eq__` —
ровно то, что иначе пишут руками. Флаги управляют поведением: `order=True`
добавляет сравнения, `frozen=True` запрещает менять поля. `Enum` — тип с
фиксированным набором значений-одиночек:

In [38]:
from dataclasses import dataclass, field
from enum import Enum


@dataclass(frozen=True, order=True)
class Case:
    name: str
    given: int
    s: int = field(default=10)


class DeviceType(Enum):
    ANDROID = "android"
    WINDOWS = "windows"
    MACOS = "macos"


case = Case("test_name", 2)
print(case)
print(Case("a", 1) < Case("b", 1), case.s)

d1, d2 = DeviceType.MACOS, DeviceType.MACOS
print(d1 is d2, d1.name, d1.value, {d1: "ключ"})

Case(name='test_name', given=2, s=10)
True 10
True MACOS macos {<DeviceType.MACOS: 'macos'>: 'ключ'}


---

## Часть 2. Углубление

### 2.0. `__repr__` подменяет `__str__`, но не наоборот

`print`, `str()` и f-строка зовут `__str__`; если его нет, Python берёт
`__repr__`. Обратное не работает: один `__str__` не спасёт ни отладчик, ни
контейнер. Поэтому если писать один метод — это `__repr__`:

In [40]:
class OnlyRepr:
    def __repr__(self) -> str:
        return "OnlyRepr(тип и содержимое)"

class OnlyStr:
    def __str__(self) -> str:
        return "OnlyStr(тип и содержимое)"

print(OnlyRepr())          # __str__ нет — его подменил __repr__
print(str(OnlyRepr()))     # str() берёт тот же __repr__
print(f"{OnlyRepr()}")     # и f-строка тоже

print(OnlyStr())          
print([OnlyStr()])          # __str__ нет — его подменил __repr__

OnlyRepr(тип и содержимое)
OnlyRepr(тип и содержимое)
OnlyRepr(тип и содержимое)
OnlyStr(тип и содержимое)


Раздел 1 семинара; задача семинара `printing.py`, ДЗ `date_factory` — там
нужны оба: `Date(day=5, month=3, year=2026)` для отладки и `05.03.2026` для
читателя.

### 2.1. Контейнер печатает элементы через `__repr__`

Список словарей с вашим объектом внутри не спрашивают про `__str__` —
элементы всегда идут через `__repr__`, даже когда красивый `__str__`
определён. Быстрая привычка — проверять своё представление именно в списке:

In [43]:
class Book:
    def __init__(self, title: str, year: int) -> None:
        self.title, self.year = title, year

    def __repr__(self) -> str:
        return f"Book('{self.title}', {self.year})"

    def __str__(self) -> str:
        return f"{self.title} ({self.year})"


shelf = [Book("Мастер", 1967), Book("Мы", 1920)]
for book in shelf:
    print(book)            # __str__ — для читателя
print(shelf)               # элементы списка — через __repr__

Мастер (1967)
Мы (1920)
[Book('Мастер', 1967), Book('Мы', 1920)]


Оба метода из одной пары — разный адресат: `__repr__` для разработчика
(видно тип и поля), `__str__` для вывода программы.

In [ ]:
class Coord:
    def __init__(self, lat: float, lon: float) -> None:
        self.lat, self.lon = lat, lon

    def __eq__(self, other: object) -> bool:
        return (
            isinstance(other, Coord)
            and (self.lat, self.lon) == (other.lat, other.lon)
        )


print(Coord(1.0, 2.0) == Coord(1.0, 2.0))
print(Coord.__hash__)

try:
    len({Coord(1.0, 2.0), Coord(1.0, 2.0)})
except TypeError as e:
    print("TypeError:", e)

In [ ]:
class Coord:
    def __init__(self, lat: float, lon: float) -> None:
        self.lat, self.lon = lat, lon

    def __eq__(self, other: object) -> bool:
        return (
            isinstance(other, Coord)
            and (self.lat, self.lon) == (other.lat, other.lon)
        )

    def __hash__(self) -> int:
        return hash((self.lat, self.lon))   # те же поля, что в __eq__


points = {Coord(1.0, 2.0), Coord(1.0, 2.0), Coord(3.0, 4.0)}
print(len(points))

Хеш — из неизменяемого кортежа тех же полей, что участвуют в равенстве
(контракт подробно — в Prep5, 2.9). Второй путь — `@dataclass(frozen=True)`:
сгенерирует и равенство, и хеш, и запретит менять поля. Раздел 2 семинара;
задача семинара `hashable.py`, ДЗ `date_factory`, `geo_point`.

### 2.3. `NotImplemented` — это не `NotImplementedError`

`NotImplementedError` — исключение «метод не дописан». `NotImplemented` —
значение, которое dunder-метод возвращает: «этот операнд не мой». Получив его
от первого операнда, Python не падает, а спрашивает reflected-метод второго,
и только после второго отказа поднимает `TypeError`:

In [44]:
from typing import Self


class Money:
    def __init__(self, kopecks: int) -> Self:
        self.kopecks = kopecks

    def __add__(self, other: object) -> Self:
        if isinstance(other, Money):
            return Money(self.kopecks + other.kopecks)
        return NotImplemented             # не умею — спроси его

    def __radd__(self, other: int) -> Self:
        print("спросили __radd__")
        return Money(self.kopecks + other * 100)

    def __repr__(self) -> str:
        return f"Money({self.kopecks})"


price = Money(2500)
print(price + Money(100))     # оба Money — обычный путь
print(2 + price)              # int не умеет + Money → наш __radd__

Money(2600)
спросили __radd__
Money(2700)


Вот кто напечатал вторую сумму: не мы, а reflected-метод. В ДЗ `money` на
чужой валюте арифметика возвращают `NotImplemented` (итог — `TypeError`),
а `==` честно даёт `False`. Раздел про reflected — 2.5.

### 2.4. `+` строит новый объект, `+=` мутирует старый

`a = a + b` и `a += b` — разные вызовы: `__add__` обязан вернуть новый объект,
`__iadd__` — изменить `self` и вернуть его. Если `__iadd__` не определить,
`+=` молча выродится в `a = a + b`, как у чисел. Свидетель — `id()`:

In [45]:
from typing import Self


class Money:
    def __init__(self, kopecks: int) -> Self:
        self.kopecks = kopecks

    def __add__(self, other: "Money") -> "Money":
        return Money(self.kopecks + other.kopecks)   # новый объект

    def __iadd__(self, other: "Money") -> "Money":
        self.kopecks += other.kopecks                # мутация
        return self                                  # и вернуть self

    def __repr__(self) -> str:
        return f"Money({self.kopecks})"


wallet = Money(5000)
alias = wallet
wallet_id = id(wallet)

wallet += Money(500)
print(wallet, alias, id(wallet) == wallet_id)

wallet = wallet + Money(500)
print(wallet, alias, id(wallet) == wallet_id)

Money(5500) Money(5500) True
Money(6000) Money(5500) False


Списки ведут себя как второй случай: `lst += [x]` видит каждый, у кого есть
имя на объект (Prep5, 2.0). Правило i-методов — из лекции (1.5). В ДЗ `money`
нужны обе версии: `__add__` и `__iadd__`.

### 2.5. `__mul__` и `__rmul__`: кто кого спросил

`a * b` идёт в `a.__mul__(b)`. Получив `NotImplemented`, Python разворачивает
порядок: `b.__rmul__(a)`. Поэтому `2 * obj` и `obj * 2` — разные методы,
хотя результат обязан совпадать. Метки-принты, как в лекции с классами `A`
и `B`:

In [ ]:
from typing import Self


class Scalar:
    def __init__(self, value: int) -> Self:
        self.value = value

    def __mul__(self, other: int) -> Self:
        print("Scalar.__mul__")
        return Scalar(self.value * other)

    def __rmul__(self, other: int) -> Self:
        print("Scalar.__rmul__")
        return Scalar(other * self.value)

    def __repr__(self) -> str:
        return f"Scalar({self.value})"


print(Scalar(6) * 3)
print(3 * Scalar(6))

`int` не знает, как умножаться на `Scalar`, — во второй строке работа ушла в
наш `__rmul__`.

### 2.6. `__slots__` у наследника

Слоты действуют только на свой класс. Наследник, не объявивший `__slots__`,
получает обратно `__dict__` — а с ним возможность вешать любые атрибуты; обе
выгоды слотов исчезают. Лечится объявлением на каждом уровне, хотя бы пустым
кортежем:

In [46]:
class Tick:
    __slots__ = ("symbol", "price")


class TradeTick(Tick):          # забыл про слоты
    pass


trade = TradeTick()
trade.side = "buy"              # сработало!
print(hasattr(trade, "__dict__"))


class StrictTick(Tick):
    __slots__ = ("side",)       # новые поля — своими слотами


strict = StrictTick()
strict.side = "sell"
print(hasattr(strict, "__dict__"), strict.side)

try:
    strict.volume = 100
except AttributeError as e:
    print("AttributeError:", e)

True
False sell
AttributeError: 'StrictTick' object has no attribute 'volume' and no __dict__ for setting new attributes




### 2.7. Элемент `Enum` не равен своему значению

Элемент перечисления — самостоятельный объект: `Device.MAC == "macos"` даёт
`False`, и ошибки не будет — условие молча не сработает. Строку превращают в
элемент на входе: круглые скобки ищут по значению, квадратные — по имени.
Два имени с одним значением — не два элемента, а псевдоним:

In [47]:
from enum import Enum


class Device(Enum):
    MAC = "macos"
    WIN = "windows"


class Platform(Enum):
    MAC = "macos"
    APPLE = "macos"             # псевдоним, не новый элемент


print(Device.MAC == "macos", Device.MAC.value == "macos")
print(Device("macos"), Device["WIN"])
print(Platform.APPLE is Platform.MAC, [p.name for p in Platform])

False True
Device.MAC Device.WIN
True ['MAC']


Сравнивайте значения через `.value` — или сразу нормализуйте строку в
элемент.
### 2.8. `dataclass`: изменяемый дефолт и флаги

Список в дефолте поля стал бы общим для всех объектов — та же ловушка, что у
функций (Prep5, 2.3), только в классе. `dataclass` про неё знает и отказывается
создавать класс. Фикс — `field(default_factory=…)`: новый список каждому.
Комбинация `frozen=True, order=True` — готовый рецепт неизменяемой
сортируемой записи:

In [48]:
from dataclasses import dataclass, field


try:

    @dataclass
    class Basket:
        items: list[str] = []

except ValueError as e:
    print("ValueError при объявлении:", e)


@dataclass
class Report:
    tags: list[str] = field(default_factory=list)   # свой список каждому


report = Report()
report.tags.append("утро")
print(Report().tags, report.tags)


@dataclass(frozen=True, order=True)
class Measurement:
    moment: str
    value: float


print(Measurement("09:00", 1.5) < Measurement("21:00", 0.5))
print(sorted([Measurement("21:00", 0.5), Measurement("09:00", 1.5)]))

ValueError при объявлении: mutable default <class 'list'> for field items is not allowed: use default_factory
[] ['утро']
True
[Measurement(moment='09:00', value=1.5), Measurement(moment='21:00', value=0.5)]


Ошибка выстреливает при **объявлении класса**, ещё до первого объекта — её
невозможно не заметить. Задача семинара `records.py` (`Measurement` с
`frozen=True, order=True`: `latest`, `distinct`, `in_order`), ДЗ `geo_point` —
тот же набор флагов. Раздел 5 семинара.

### 2.9. `property`: одно данное, три шкалы

`property` раскрывается в полную силу, когда одно хранимое данное нужно
показывать в нескольких видах. Хранить три поля значило бы держать три копии,
способные разъехаться. Храним цельсии, остальное пересчитываем, — а проверка
абсолютного нуля живёт в одном месте, и все сеттеры пишут через `celsius`:

In [49]:
class Temperature:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius           # уже вызов set_celsius с проверкой

    def get_celsius(self) -> float:
        return self._celsius

    def set_celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError("ниже абсолютного нуля")
        self._celsius = value

    celsius = property(fget=get_celsius, fset=set_celsius)

    def get_fahrenheit(self) -> float:
        return self._celsius * 9 / 5 + 32

    def set_fahrenheit(self, value: float) -> None:
        self.celsius = (value - 32) * 5 / 9    # через celsius — одна проверка

    fahrenheit = property(fget=get_fahrenheit, fset=set_fahrenheit)

    def get_kelvin(self) -> float:
        return self._celsius + 273.15

    def set_kelvin(self, value: float) -> None:
        self.celsius = value - 273.15

    kelvin = property(fget=get_kelvin, fset=set_kelvin)


t = Temperature(25.0)
print(t.celsius, t.fahrenheit, t.kelvin)

t.fahrenheit = 212          # пишем фаренгейтами…
print(t.celsius)            # …а хранятся всё те же цельсии

try:
    t.kelvin = -5           # холоднее нуля нельзя ни в одной шкале
except ValueError as e:
    print("ValueError:", e)

25.0 77.0 298.15
100.0
ValueError: ниже абсолютного нуля


Одна копия истины, три вида наружу, валидация в одном месте — ДЗ `temperature`
ровно об этом. Форма записи `property(fget=…, fset=…)` — из лекции (1.6);
декоратором то же самое пишут короче, но это тема лекции про декораторы.

### 2.10. Фабрика переиспользует валидацию конструктора

Ценность фабрики из 1.1 — не в количестве способов создать объект, а в том,
что проверка живёт в одном месте: фабрика разбирает вход и отдаёт результат
обычному `__init__`. Невалидный вход обрабатывается одинаково во всех путях
создания:

In [ ]:
from enum import Enum


class Status(Enum):
    NEW = "new"
    PAID = "paid"


def parse(raw: str) -> Status | None:
    try:
        return Status(raw)               # ищем по значению
    except ValueError:
        return None                      # семинар без исключений: вернём None


print(parse("paid"), parse("cancelled"))


class Date:
    def __init__(self, day: int, month: int, year: int) -> None:
        if not 1 <= month <= 12:
            raise ValueError("month out of range")
        self.day, self.month, self.year = day, month, year

    def __repr__(self) -> str:
        return f"Date(day={self.day}, month={self.month}, year={self.year})"

    @staticmethod
    def from_iso(iso: str) -> "Date":
        year, month, day = map(int, iso.split("-"))
        return Date(day, month, year)    # та же проверка, что и везде


print(Date.from_iso("2026-03-05"))

try:
    Date.from_iso("2026-13-05")
except ValueError as e:
    print("ValueError:", e)

Один конструктор с проверкой, много фабрик вокруг него. В семинаре невалидный
вход — `None` (исключения — тема лекции 7), в ДЗ `date_factory` и `market_tick`
фабрики (`from_iso`, `from_american`, `from_csv`) работают поверх той же
валидации. Задача семинара `statuses.py` (`parse`, `parse_all`).

---

## Часть 3. Викторина «предскажи, что выведет»

Десять вопросов — по recap'у и углублению: печать, операторы, слоты, enum и
dataclass. Порядок действий: предскажи вывод, включая исключение, если оно
будет → запусти ячейку → раскрой разбор. 10 из 10 без единого запуска — к
семинару готов.

### Вопрос 1

У класса определили только `__repr__`. Что выведут оба вызова?

In [50]:
class Shadow:
    def __repr__(self) -> str:
        return "Shadow(репрезентация)"


print(Shadow())
print(str(Shadow()))

Shadow(репрезентация)
Shadow(репрезентация)


<details>
<summary><b>Разбор</b></summary>

`Shadow(репрезентация)` дважды. `print` и `str()` зовут `__str__`, но его
нет — и его место занимает `__repr__` (2.0). Обратная подмена не работает:
класс только с `__str__` внутри контейнера печатается унылым дефолтом (2.1).

</details>

### Вопрос 2

У книги есть оба метода. Что выведет программа?

In [55]:
class Book:
    def __repr__(self) -> str:
        return "Book(repr)"

    def __str__(self) -> str:
        return "Book(str)"


shelf = [Book(), Book()]
print(shelf[0])
print(shelf)

Book(str)
[Book(repr), Book(repr)]


<details>
<summary><b>Разбор</b></summary>

`Book(str)`, затем `[Book(repr), Book(repr)]`. Одиночный объект печатается
через `__str__`, а контейнер своих элементов спрашивает всегда через
`__repr__` — есть `__str__` или нет (2.1). Задача семинара `printing.py`.

</details>

### Вопрос 3

Первая строка печатает результат. Что случится со второй?

In [56]:
class Point:
    def __init__(self, x: int) -> None:
        self.x = x

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Point) and self.x == other.x


print(Point(1) == Point(1))
print(len({Point(1), Point(1)}))

True


TypeError: cannot use 'Point' as a set element (unhashable type: 'Point')

<details>
<summary><b>Разбор</b></summary>

`True`, затем `TypeError: cannot use 'Point' as a set element
(unhashable type: 'Point')`. Своё равенство по
полям затирает унаследованный `__hash__` (2.2): Python не может гарантировать
контракт «равные → равные хеши». Фикс — вернуть `__hash__` из кортежа полей
или взять `@dataclass(frozen=True)`. Задача семинара `hashable.py`.

</details>

### Вопрос 4

`__add__` умеет только свой тип. Эта ячейка падает — где именно?

In [57]:
class Number:
    def __init__(self, x: int) -> None:
        self.x = x

    def __repr__(self) -> str:
        return f"Number({self.x})"

    def __add__(self, other: object) -> "Number":
        if isinstance(other, Number):
            return Number(self.x + other.x)
        return NotImplemented


print(Number(2) + Number(3))
print(Number(2) + "мимо")

Number(5)


TypeError: unsupported operand type(s) for +: 'Number' and 'str'

<details>
<summary><b>Разбор</b></summary>

`Number(5)` печатается, а вторая строка поднимает `TypeError: unsupported
operand type(s) for +: 'Number' and 'str'`. Наш `__add__` вернул
`NotImplemented` — значение, не исключение (2.3); Python спросил отражённый
метод строки, там тоже отказ — и только тогда `TypeError`.

</details>

### Вопрос 5

`+` и `+=` с одними и теми же числами. Что напечатается?

In [ ]:
class Money:
    def __init__(self, kopecks: int) -> None:
        self.kopecks = kopecks

    def __add__(self, other: "Money") -> "Money":
        return Money(self.kopecks + other.kopecks)

    def __iadd__(self, other: "Money") -> "Money":
        self.kopecks += other.kopecks
        return self

    def __repr__(self) -> str:
        return f"Money({self.kopecks})"


wallet = Money(1000)
alias = wallet

wallet += Money(500)
print(wallet, alias, wallet is alias)

wallet = wallet + Money(500)
print(wallet, alias, wallet is alias)

<details>
<summary><b>Разбор</b></summary>

`Money(1500) Money(1500) True`, затем `Money(2000) Money(1500) False`.
`+=` пошёл в `__iadd__` — мутация и возврат `self` (2.4): оба имени смотрят
на один объект. Обычный `+` построил новый объект, `alias` остался при
старом.

</details>

### Вопрос 6

Два умножения и принты-метки. Какие строки и в каком порядке?

In [58]:
class Scalar:
    def __init__(self, value: int) -> None:
        self.value = value

    def __mul__(self, other: int) -> "Scalar":
        print("__mul__")
        return Scalar(self.value * other)

    def __rmul__(self, other: int) -> "Scalar":
        print("__rmul__")
        return Scalar(other * self.value)

    def __repr__(self) -> str:
        return f"Scalar({self.value})"


print(Scalar(6) * 3)
print(3 * Scalar(6))

__mul__
Scalar(18)
__rmul__
Scalar(18)


<details>
<summary><b>Разбор</b></summary>

`__mul__`, `Scalar(18)`, `__rmul__`, `Scalar(18)`. Первый вызов — прямой
`__mul__`. Во втором `int` с `Scalar` не справился, вернул `NotImplemented` —
и Python развернул порядок: сработал наш `__rmul__` (2.5). Результат тот же,
путь другой.

</details>

### Вопрос 7

In [59]:
class Tick:
    __slots__ = ("symbol", "price")


tick = Tick()
tick.symbol = "AAPL"
tick.cost = 100

AttributeError: 'Tick' object has no attribute 'cost' and no __dict__ for setting new attributes

<details>
<summary><b>Разбор</b></summary>

Последняя строка: `AttributeError: 'Tick' object has no attribute
'cost' and no __dict__ for setting new attributes`. Слоты перечислили только
`symbol` и `price`, персонального `__dict__` у объекта нет — дописать поле
некуда (1.4). ДЗ `market_tick`: опечатка в имени поля должна ловиться именно
так.

</details>

### Вопрос 8

Наследник слотового класса без своих `__slots__`. Что напечатается?

In [60]:
class Base:
    __slots__ = ("x",)


class Child(Base):
    pass


child = Child()
child.x = 1
child.anything = 2

print(hasattr(child, "__dict__"), child.anything)

True 2


<details>
<summary><b>Разбор</b></summary>

`True 2`. Слоты действуют только на свой класс (2.6): наследник без
объявления получил `__dict__` обратно, и любые атрибуты снова разрешены —
экономия памяти закончилась на первом же наследнике. Фикс — `__slots__ = ()`
в каждом наследнике.

</details>

### Вопрос 9

Элемент перечисления против его строкового значения. Что выведется?

In [61]:
from enum import Enum


class Device(Enum):
    MAC = "macos"
    WIN = "windows"


print(Device.MAC == "macos")
print(Device("macos") is Device.MAC)
print(Device["WIN"])
print(Device["WIN"].name)
print(Device["WIN"].value)

False
True
Device.WIN
WIN
windows


<details>
<summary><b>Разбор</b></summary>

`False`, `True`, `Device.WIN`. Элемент — самостоятельный объект и строке
не равен (2.7); `Device("macos")` ищет по значению и возвращает того же
элемента-одиночку; квадратные скобки — поиск по имени. Печать элемента —
`Класс.Имя`. Задача семинара `statuses.py`.

</details>

### Вопрос 10

In [62]:
from dataclasses import dataclass


@dataclass(frozen=True)
class GeoPoint:
    latitude: float
    longitude: float


point = GeoPoint(55.75, 37.62)
point.latitude = 0.0

print(point)

FrozenInstanceError: cannot assign to field 'latitude'

<details>
<summary><b>Разбор</b></summary>

`dataclasses.FrozenInstanceError: cannot assign to field 'latitude'` —
падает присваивание, до `print` дело не доходит. `frozen=True` запрещает
менять поля после создания (1.8, 2.8), зато такой класс хешируем — годится
ключом словаря. ДЗ `geo_point`.

</details>